In [1]:
# Standard library
import os
import sys
import glob
import shutil
import subprocess
import importlib
from pathlib import Path

# Data handling
import pandas as pd

# PySpark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
    BooleanType, TimestampType
)

In [21]:
PROJECT_ROOT = Path(r"D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction")

# Source code (existing ingestion script lives here)
SRC_DIR = PROJECT_ROOT / "src" / "ingestion"
INGESTION_SCRIPT = SRC_DIR / "process_raw_data.py"

# Data directories
PROCESSED_DIR = PROJECT_ROOT / "data" / "raw_data_csv"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
DATABASE_DIR = PROJECT_ROOT / "data" / "database"

for d in (RAW_DIR, PROCESSED_DIR, CLEANED_DIR, DATABASE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# The six flat CSV files produced by process_raw_data.py from the raw BODS XML
EXPECTED_CSV_FILES = [
    "timetable_stops.csv",
    "timetable_stop_times.csv",
    "timetable_vehicle_journeys.csv",
    "disruptions.csv",
    "fares.csv",
    "location_pings.csv",
]

print(f"Project root:      {PROJECT_ROOT}")
print(f"Ingestion script:  {INGESTION_SCRIPT}")
print(f"Processed dir:     {PROCESSED_DIR}")
print(f"Cleaned dir:       {CLEANED_DIR}")
print(f"Database dir:      {DATABASE_DIR}")


Project root:      D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction
Ingestion script:  D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\src\ingestion\process_raw_data.py
Processed dir:     D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\raw_data_csv
Cleaned dir:       D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned
Database dir:      D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\database


In [3]:
N_CORES = 8

spark = (
    SparkSession.builder
    .appName("BusDelay_DataCollectionCleaning")
    .master(f"local[{N_CORES}]")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 2))
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version:        {spark.version}")
print(f"Cores configured:      {N_CORES}")
print(f"Default parallelism:   {spark.sparkContext.defaultParallelism}")


Spark version:        3.5.8
Cores configured:      8
Default parallelism:   8


In [4]:
def missing_csv_files():
    '''Return the list of expected CSVs that are not yet in data/raw_data_csv/.'''
    return [f for f in EXPECTED_CSV_FILES if not (PROCESSED_DIR / f).exists()]


missing = missing_csv_files()

if missing:
    print(f"Missing {len(missing)} file(s): {missing}")
    print(f"Running existing ingestion script: {INGESTION_SCRIPT}\n")

    if not INGESTION_SCRIPT.exists():
        raise FileNotFoundError(
            f"process_raw_data.py not found at {INGESTION_SCRIPT}. "
            "Update INGESTION_SCRIPT / SRC_DIR above to match your project layout."
        )

    result = subprocess.run(
        [sys.executable, str(INGESTION_SCRIPT)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("process_raw_data.py exited with an error -- see output above.")

    still_missing = missing_csv_files()
    if still_missing:
        raise RuntimeError(
            f"Ingestion script ran but these files are still missing: {still_missing}"
        )
    print("\nIngestion script completed successfully -- all expected CSVs now present.")
else:
    print("All expected CSV files already exist in data/raw_data_csv/ -- "
          "skipping re-execution of process_raw_data.py.")


All expected CSV files already exist in data/raw_data_csv/ -- skipping re-execution of process_raw_data.py.


In [5]:
# Verification: confirm every expected file is present, non-empty, and report size
print(f"{'File':<35}{'Exists':<10}{'Size (KB)':<12}{'Rows (incl. header)'}")
print("-" * 85)
for fname in EXPECTED_CSV_FILES:
    fpath = PROCESSED_DIR / fname
    exists = fpath.exists()
    size_kb = round(fpath.stat().st_size / 1024, 1) if exists else 0
    if exists:
        with open(fpath, "r", encoding="utf-8", errors="ignore") as fh:
            n_lines = sum(1 for _ in fh)
    else:
        n_lines = "-"
    print(f"{fname:<35}{str(exists):<10}{size_kb:<12}{n_lines}")

assert not missing_csv_files(), "Some expected CSV files are still missing after verification."
print("\nAll six ingested CSV files verified.")


File                               Exists    Size (KB)   Rows (incl. header)
-------------------------------------------------------------------------------------
timetable_stops.csv                True      2745.5      20571
timetable_stop_times.csv           True      168143.7    1126332
timetable_vehicle_journeys.csv     True      5676.5      35998
disruptions.csv                    True      121.0       722
fares.csv                          True      707.4       3436
location_pings.csv                 True      6086.6      21049

All six ingested CSV files verified.


In [6]:
def standardize_column_names(df):
    '''Lowercase, strip, and underscore-ify every column name.'''
    for c in df.columns:
        clean_name = c.strip().lower().replace(" ", "_")
        if clean_name != c:
            df = df.withColumnRenamed(c, clean_name)
    return df

In [7]:
def trim_string_columns(df):
    '''Trim whitespace on every string column and convert empty-after-trim
    strings into real nulls so missing-value checks catch them.'''
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, F.trim(F.col(c)))
        df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
    return df

In [8]:
def missing_value_report(df, label=""):
    '''Print the count/percentage of nulls per column.'''
    n = df.count()
    report = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    print(f"\nMissing value report -- {label} (of {n:,} rows):")
    any_missing = False
    for col_name, missing_count in report.items():
        if missing_count > 0:
            any_missing = True
            pct = 100 * missing_count / n if n else 0
            print(f"  {col_name:<30}: {missing_count:>10,} ({pct:.1f}%)")
    if not any_missing:
        print("  No missing values found.")
    return report

In [9]:
def duplicate_count(df, subset, label=""):
    '''Print and return the number of duplicate rows on a given key.'''
    total = df.count()
    distinct = df.dropDuplicates(subset).count()
    removed = total - distinct
    print(f"Duplicate check -- {label} on {subset}: "
          f"{removed:,} duplicate row(s) found ({total:,} -> {distinct:,})")
    return removed

In [10]:
class DQTracker:
    '''Collects before/after row counts for the final data-quality summary table.'''

    def __init__(self, name):
        self.name = name
        self.original_rows = None
        self.after_dedup_rows = None
        self.after_missing_rows = None
        self.final_rows = None

    def as_row(self):
        dup_removed = (self.original_rows - self.after_dedup_rows
                       if self.original_rows is not None and self.after_dedup_rows is not None else 0)
        missing_handled = (self.after_dedup_rows - self.after_missing_rows
                            if self.after_dedup_rows is not None and self.after_missing_rows is not None else 0)
        invalid_removed = (self.after_missing_rows - self.final_rows
                            if self.after_missing_rows is not None and self.final_rows is not None else 0)
        return {
            "Dataset": self.name,
            "Original Rows": self.original_rows,
            "Final Rows": self.final_rows,
            "Duplicates Removed": dup_removed,
            "Missing Handled": missing_handled,
            "Invalid Removed": invalid_removed,
        }

In [11]:
def save_single_csv(df, out_dir: Path, filename: str, n_rows_hint=None):
    '''Write a Spark DataFrame out as a single, cleanly-named CSV file
    (Spark normally writes a folder of part-files -- this collapses that
    down to one file with the exact name we want).'''
    output_folder = out_dir / f"_{filename}_tmp"
    final_csv = out_dir / filename

    (
        df.coalesce(1).write
        .mode("overwrite")
        .option("header", True)
        .csv(str(output_folder))
    )

    csv_file = glob.glob(str(output_folder / "part-*.csv"))[0]
    if final_csv.exists():
        final_csv.unlink()
    shutil.move(csv_file, str(final_csv))
    shutil.rmtree(output_folder)

    row_note = f"{n_rows_hint:,} rows" if n_rows_hint is not None else ""
    print(f"Saved -> {final_csv} ({row_note})")


# Collects every dataset's DQTracker so the summary section can build the
# final comparison table without re-running anything.
dq_trackers = []
print("Helper functions and DQTracker ready.")

Helper functions and DQTracker ready.


In [12]:
stops_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("stop_point_ref", StringType(), True),
    StructField("common_name", StringType(), True),
])

try:
    stops_raw = spark.read.csv(
        str(PROCESSED_DIR / "timetable_stops.csv"), header=True, schema=stops_schema
    )
except Exception as e:
    raise SystemExit(f"Failed to load timetable_stops.csv: {e}")

tracker_stops = DQTracker("timetable_stops")
tracker_stops.original_rows = stops_raw.count()
print(f"Loaded {tracker_stops.original_rows:,} rows")


Loaded 20,570 rows


In [13]:
print(f"Row count: {stops_raw.count():,}")
print(f"Columns:   {stops_raw.columns}")
stops_raw.printSchema()
stops_raw.show(5, truncate=False)


Row count: 20,570
Columns:   ['source_file', 'stop_point_ref', 'common_name']
root
 |-- source_file: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- common_name: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------+--------------+----------------------------+
|source_file                                                                                                |stop_point_ref|common_name                 |
+-----------------------------------------------------------------------------------------------------------+--------------+----------------------------+
|1-None--SCCM-CACA-2025-06-01-Live_CB_2025_06_01_-_School_back_+_C__SCCM_PF0000459_27_20250619-BODS_V1_1.xml|0500CCITY242  |Arbury Campkin Rd           |
|1-None--SCCM-CACA-2025-06-01-Live_CB_2025_06_01_-_School_back_+_C__SCCM_PF0000459_27_20250619-BODS_V1_1.xml|0500CCITY334  |Kings Hedges Hawkins Road   |
|1-None--SCCM-

In [14]:
stops_std = standardize_column_names(stops_raw)
stops_std = trim_string_columns(stops_std)

missing_value_report(stops_std, "timetable_stops (raw)")
duplicate_count(stops_std, ["stop_point_ref"], "timetable_stops")

invalid_empty_ref = stops_std.filter(F.col("stop_point_ref").isNull()).count()
print(f"\nInvalid records -- null/empty stop_point_ref: {invalid_empty_ref:,}")



Missing value report -- timetable_stops (raw) (of 20,570 rows):
  No missing values found.
Duplicate check -- timetable_stops on ['stop_point_ref']: 11,675 duplicate row(s) found (20,570 -> 8,895)

Invalid records -- null/empty stop_point_ref: 0


In [15]:
# Clean: de-duplicate on the natural key, drop unusable rows,
stops_clean = stops_std.dropDuplicates(["stop_point_ref"])
tracker_stops.after_dedup_rows = stops_clean.count()

stops_clean = stops_clean.filter(F.col("stop_point_ref").isNotNull())
stops_clean = stops_clean.fillna({"common_name": "UNKNOWN_STOP_NAME"})
tracker_stops.after_missing_rows = stops_clean.count()

# No numeric-range validation applies to this dataset.
tracker_stops.final_rows = stops_clean.count()
print(f"Rows after cleaning: {tracker_stops.final_rows:,}")


Rows after cleaning: 8,895


In [16]:
missing_value_report(stops_clean, "timetable_stops (cleaned)")
duplicate_count(stops_clean, ["stop_point_ref"], "timetable_stops (cleaned)")

dq_trackers.append(tracker_stops)
print(tracker_stops.as_row())


Missing value report -- timetable_stops (cleaned) (of 8,895 rows):
  No missing values found.
Duplicate check -- timetable_stops (cleaned) on ['stop_point_ref']: 0 duplicate row(s) found (8,895 -> 8,895)
{'Dataset': 'timetable_stops', 'Original Rows': 20570, 'Final Rows': 8895, 'Duplicates Removed': 11675, 'Missing Handled': 0, 'Invalid Removed': 0}


In [20]:
save_single_csv(stops_clean, CLEANED_DIR, "timetable_stops.csv", tracker_stops.final_rows)


Saved -> D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned\timetable_stops.csv (8,895 rows)


In [22]:
vj_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("vehicle_journey_code", StringType(), True),
    StructField("service_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("line_name", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("journey_pattern_ref", StringType(), True),
    StructField("scheduled_departure_time", StringType(), True),
])

try:
    vj_raw = spark.read.csv(
        str(PROCESSED_DIR / "timetable_vehicle_journeys.csv"), header=True, schema=vj_schema
    )
except Exception as e:
    raise SystemExit(f"Failed to load timetable_vehicle_journeys.csv: {e}")

tracker_vj = DQTracker("timetable_vehicle_journeys")
tracker_vj.original_rows = vj_raw.count()
print(f"Loaded {tracker_vj.original_rows:,} rows")


Loaded 35,997 rows


In [23]:
print(f"Row count: {vj_raw.count():,}")
print(f"Columns:   {vj_raw.columns}")
vj_raw.printSchema()
vj_raw.show(5, truncate=False)


Row count: 35,997
Columns:   ['source_file', 'vehicle_journey_code', 'service_ref', 'line_ref', 'line_name', 'operator_ref', 'journey_pattern_ref', 'scheduled_departure_time']
root
 |-- source_file: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- service_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- journey_pattern_ref: string (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------+--------------------+------------+-------------------+---------+------------+-------------------+------------------------+
|source_file                                                                                                |vehicle_journey_code|service_ref |line_ref           |line_name|operator_ref|journey_pattern_ref|scheduled_

In [24]:
vj_std = standardize_column_names(vj_raw)
vj_std = trim_string_columns(vj_std)

missing_value_report(vj_std, "timetable_vehicle_journeys (raw)")
duplicate_count(vj_std, ["vehicle_journey_code", "source_file"], "timetable_vehicle_journeys")

bad_time_pattern = ~F.col("scheduled_departure_time").rlike(r"^\d{1,2}:\d{2}:\d{2}$")
n_bad_time = vj_std.filter(
    F.col("scheduled_departure_time").isNotNull() & bad_time_pattern
).count()
print(f"\nInvalid records -- malformed scheduled_departure_time pattern: {n_bad_time:,}")



Missing value report -- timetable_vehicle_journeys (raw) (of 35,997 rows):
  No missing values found.
Duplicate check -- timetable_vehicle_journeys on ['vehicle_journey_code', 'source_file']: 0 duplicate row(s) found (35,997 -> 35,997)

Invalid records -- malformed scheduled_departure_time pattern: 0


In [25]:
vj_clean = vj_std.dropDuplicates(["vehicle_journey_code", "source_file"])
tracker_vj.after_dedup_rows = vj_clean.count()

vj_clean = vj_clean.filter(
    F.col("vehicle_journey_code").isNotNull() &
    F.col("line_ref").isNotNull() &
    F.col("scheduled_departure_time").isNotNull()
)
vj_clean = vj_clean.fillna({"line_name": "UNKNOWN_LINE_NAME"})
tracker_vj.after_missing_rows = vj_clean.count()

vj_clean = vj_clean.withColumn(
    "scheduled_departure_ts",
    F.try_to_timestamp(F.concat(F.lit("1970-01-01 "), F.col("scheduled_departure_time")))
)
vj_clean = vj_clean.filter(F.col("scheduled_departure_ts").isNotNull())
tracker_vj.final_rows = vj_clean.count()
print(f"Rows after cleaning: {tracker_vj.final_rows:,}")

Rows after cleaning: 35,997


In [26]:
missing_value_report(vj_clean, "timetable_vehicle_journeys (cleaned)")
duplicate_count(vj_clean, ["vehicle_journey_code", "source_file"], "timetable_vehicle_journeys (cleaned)")

print("\nJourneys per line (top 10):")
vj_clean.groupBy("line_ref").count().orderBy(F.desc("count")).show(10)

dq_trackers.append(tracker_vj)
print(tracker_vj.as_row())



Missing value report -- timetable_vehicle_journeys (cleaned) (of 35,997 rows):
  No missing values found.
Duplicate check -- timetable_vehicle_journeys (cleaned) on ['vehicle_journey_code', 'source_file']: 0 duplicate row(s) found (35,997 -> 35,997)

Journeys per line (top 10):
+--------------------+-----+
|            line_ref|count|
+--------------------+-----+
| SCOX:PH0005863:12:1| 1352|
|SCOX:PH0005863:11...| 1118|
| SCOX:PH0005863:43:8|  852|
| SCOX:PH0005863:13:2|  846|
| SCGL:PH0005031:10:1|  750|
|SCOX:PH0005863:13:2A|  710|
|SCGL:PH0005031:13:10|  692|
|SCGL:PH0005031:182:C|  668|
|SCGL:PH0005031:180:A|  664|
|SCOX:PH0005863:16:S1|  618|
+--------------------+-----+
only showing top 10 rows

{'Dataset': 'timetable_vehicle_journeys', 'Original Rows': 35997, 'Final Rows': 35997, 'Duplicates Removed': 0, 'Missing Handled': 0, 'Invalid Removed': 0}


In [27]:
save_single_csv(vj_clean, CLEANED_DIR, "timetable_vehicle_journeys.csv", tracker_vj.final_rows)


Saved -> D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned\timetable_vehicle_journeys.csv (35,997 rows)


In [28]:
stop_times_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("vehicle_journey_code", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("stop_point_ref", StringType(), True),
    StructField("stop_sequence", IntegerType(), True),
    StructField("scheduled_time", StringType(), True),
])

try:
    stop_times_raw = spark.read.csv(
        str(PROCESSED_DIR / "timetable_stop_times.csv"), header=True, schema=stop_times_schema
    )
    stop_times_raw = stop_times_raw.repartition(N_CORES * 2, "line_ref").cache()
except Exception as e:
    raise SystemExit(f"Failed to load timetable_stop_times.csv: {e}")

tracker_st = DQTracker("timetable_stop_times")
tracker_st.original_rows = stop_times_raw.count()  # materialises the cache
print(f"Loaded {tracker_st.original_rows:,} rows, "
      f"{stop_times_raw.rdd.getNumPartitions()} partitions")


Loaded 1,126,331 rows, 16 partitions


In [29]:
print(f"Row count: {stop_times_raw.count():,}")
print(f"Columns:   {stop_times_raw.columns}")
stop_times_raw.printSchema()
stop_times_raw.show(5, truncate=False)

Row count: 1,126,331
Columns:   ['source_file', 'vehicle_journey_code', 'line_ref', 'stop_point_ref', 'stop_sequence', 'scheduled_time']
root
 |-- source_file: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- stop_point_ref: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- scheduled_time: string (nullable = true)

+-----------------------------------------------------------------------------------------------------+--------------------+----------------------+--------------+-------------+--------------+
|source_file                                                                                          |vehicle_journey_code|line_ref              |stop_point_ref|stop_sequence|scheduled_time|
+-----------------------------------------------------------------------------------------------------+--------------------+----------------------+--------------+-------------+--------------+
|22A-Non

In [30]:
st_std = standardize_column_names(stop_times_raw)
st_std = trim_string_columns(st_std)

missing_value_report(st_std, "timetable_stop_times (raw)")
duplicate_count(
    st_std, ["vehicle_journey_code", "stop_point_ref", "stop_sequence"],
    "timetable_stop_times"
)

n_negative_seq = st_std.filter(F.col("stop_sequence") < 0).count()
print(f"\nInvalid records -- negative stop_sequence: {n_negative_seq:,}")



Missing value report -- timetable_stop_times (raw) (of 1,126,331 rows):
  No missing values found.
Duplicate check -- timetable_stop_times on ['vehicle_journey_code', 'stop_point_ref', 'stop_sequence']: 199,850 duplicate row(s) found (1,126,331 -> 926,481)

Invalid records -- negative stop_sequence: 0


In [31]:
st_clean = st_std.dropDuplicates(
    ["vehicle_journey_code", "stop_point_ref", "stop_sequence"]
)
tracker_st.after_dedup_rows = st_clean.count()

st_clean = st_clean.filter(
    F.col("vehicle_journey_code").isNotNull() &
    F.col("stop_point_ref").isNotNull() &
    F.col("scheduled_time").isNotNull()
)
tracker_st.after_missing_rows = st_clean.count()

st_clean = st_clean.withColumn(
    "scheduled_ts",
    F.try_to_timestamp(F.concat(F.lit("1970-01-01 "), F.col("scheduled_time")))
)
st_clean = st_clean.filter(
    F.col("scheduled_ts").isNotNull() & (F.col("stop_sequence") >= 0)
)
tracker_st.final_rows = st_clean.count()
print(f"Rows after cleaning: {tracker_st.final_rows:,}")


Rows after cleaning: 926,481


In [32]:
missing_value_report(st_clean, "timetable_stop_times (cleaned)")
duplicate_count(
    st_clean, ["vehicle_journey_code", "stop_point_ref", "stop_sequence"],
    "timetable_stop_times (cleaned)"
)
print("\nSummary statistics -- stop_sequence (route length indicator):")
st_clean.describe("stop_sequence").show()

dq_trackers.append(tracker_st)
print(tracker_st.as_row())



Missing value report -- timetable_stop_times (cleaned) (of 926,481 rows):
  No missing values found.
Duplicate check -- timetable_stop_times (cleaned) on ['vehicle_journey_code', 'stop_point_ref', 'stop_sequence']: 0 duplicate row(s) found (926,481 -> 926,481)

Summary statistics -- stop_sequence (route length indicator):
+-------+------------------+
|summary|     stop_sequence|
+-------+------------------+
|  count|            926481|
|   mean|  19.9051076060923|
| stddev|15.558389601912477|
|    min|                 0|
|    max|                94|
+-------+------------------+

{'Dataset': 'timetable_stop_times', 'Original Rows': 1126331, 'Final Rows': 926481, 'Duplicates Removed': 199850, 'Missing Handled': 0, 'Invalid Removed': 0}


In [33]:
save_single_csv(
    st_clean.repartition(1), CLEANED_DIR, "timetable_stop_times.csv", tracker_st.final_rows
)


Saved -> D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned\timetable_stop_times.csv (926,481 rows)


In [34]:
location_schema = StructType([
    StructField("polled_at_utc", StringType(), True),
    StructField("recorded_at_time", StringType(), True),
    StructField("valid_until_time", StringType(), True),
    StructField("item_identifier", StringType(), True),
    StructField("vehicle_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("published_line_name", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("direction_ref", StringType(), True),
    StructField("data_frame_ref", StringType(), True),
    StructField("dated_vehicle_journey_ref", StringType(), True),
    StructField("origin_ref", StringType(), True),
    StructField("origin_name", StringType(), True),
    StructField("destination_ref", StringType(), True),
    StructField("destination_name", StringType(), True),
    StructField("origin_aimed_departure_time", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("bearing", DoubleType(), True),
    StructField("block_ref", StringType(), True),
    StructField("vehicle_journey_ref", StringType(), True),
])

try:
    location_raw = spark.read.csv(
        str(PROCESSED_DIR / "location_pings.csv"), header=True, schema=location_schema
    )
except Exception as e:
    raise SystemExit(f"Failed to load location_pings.csv: {e}")

tracker_loc = DQTracker("location_pings")
tracker_loc.original_rows = location_raw.count()
print(f"Loaded {tracker_loc.original_rows:,} rows")


Loaded 21,048 rows


In [35]:
print(f"Row count: {location_raw.count():,}")
print(f"Columns:   {location_raw.columns}")
location_raw.printSchema()
location_raw.show(5, truncate=False)


Row count: 21,048
Columns:   ['polled_at_utc', 'recorded_at_time', 'valid_until_time', 'item_identifier', 'vehicle_ref', 'line_ref', 'published_line_name', 'operator_ref', 'direction_ref', 'data_frame_ref', 'dated_vehicle_journey_ref', 'origin_ref', 'origin_name', 'destination_ref', 'destination_name', 'origin_aimed_departure_time', 'longitude', 'latitude', 'bearing', 'block_ref', 'vehicle_journey_ref']
root
 |-- polled_at_utc: string (nullable = true)
 |-- recorded_at_time: string (nullable = true)
 |-- valid_until_time: string (nullable = true)
 |-- item_identifier: string (nullable = true)
 |-- vehicle_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- published_line_name: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- direction_ref: string (nullable = true)
 |-- data_frame_ref: string (nullable = true)
 |-- dated_vehicle_journey_ref: string (nullable = true)
 |-- origin_ref: string (nullable = true)
 |-- origin_name: string (nullab

In [36]:
loc_std = standardize_column_names(location_raw)
loc_std = trim_string_columns(loc_std)

missing_value_report(loc_std, "location_pings (raw)")
duplicate_count(loc_std, ["item_identifier"], "location_pings")

n_bad_lat = loc_std.filter(~F.col("latitude").between(-90, 90)).count()
n_bad_lon = loc_std.filter(~F.col("longitude").between(-180, 180)).count()
n_bad_bearing = loc_std.filter(
    F.col("bearing").isNotNull() & ~F.col("bearing").between(0, 359)
).count()
print(f"\nInvalid records -- latitude out of range:  {n_bad_lat:,}")
print(f"Invalid records -- longitude out of range: {n_bad_lon:,}")
print(f"Invalid records -- bearing out of range:   {n_bad_bearing:,}")



Missing value report -- location_pings (raw) (of 21,048 rows):
  destination_name              :        125 (0.6%)
  origin_aimed_departure_time   :        344 (1.6%)
  bearing                       :      4,063 (19.3%)
  block_ref                     :      7,854 (37.3%)
  vehicle_journey_ref           :     20,334 (96.6%)
Duplicate check -- location_pings on ['item_identifier']: 10,521 duplicate row(s) found (21,048 -> 10,527)

Invalid records -- latitude out of range:  0
Invalid records -- longitude out of range: 0
Invalid records -- bearing out of range:   0


In [37]:
loc_clean = loc_std.dropDuplicates(["item_identifier"])
tracker_loc.after_dedup_rows = loc_clean.count()

loc_clean = loc_clean.filter(
    F.col("recorded_at_time").isNotNull() &
    F.col("latitude").isNotNull() &
    F.col("longitude").isNotNull() &
    F.col("line_ref").isNotNull()
)
tracker_loc.after_missing_rows = loc_clean.count()

loc_clean = loc_clean.withColumn(
    "recorded_ts", F.try_to_timestamp("recorded_at_time")
).withColumn(
    "origin_aimed_ts", F.try_to_timestamp("origin_aimed_departure_time")
)
loc_clean = loc_clean.filter(
    F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180)
)
loc_clean = loc_clean.withColumn(
    "bearing",
    F.when(F.col("bearing").between(0, 359), F.col("bearing")).otherwise(None)
)
tracker_loc.final_rows = loc_clean.count()
print(f"Rows after cleaning: {tracker_loc.final_rows:,}")


Rows after cleaning: 10,527


In [38]:
missing_value_report(loc_clean, "location_pings (cleaned)")
duplicate_count(loc_clean, ["item_identifier"], "location_pings (cleaned)")

print("\nSummary statistics -- latitude / longitude / bearing:")
loc_clean.describe(["latitude", "longitude", "bearing"]).show()

dq_trackers.append(tracker_loc)
print(tracker_loc.as_row())



Missing value report -- location_pings (cleaned) (of 10,527 rows):
  destination_name              :         25 (0.2%)
  origin_aimed_departure_time   :         91 (0.9%)
  bearing                       :      2,432 (23.1%)
  block_ref                     :        235 (2.2%)
  vehicle_journey_ref           :     10,411 (98.9%)
  origin_aimed_ts               :         91 (0.9%)
Duplicate check -- location_pings (cleaned) on ['item_identifier']: 0 duplicate row(s) found (10,527 -> 10,527)

Summary statistics -- latitude / longitude / bearing:
+-------+-------------------+-------------------+------------------+
|summary|           latitude|          longitude|           bearing|
+-------+-------------------+-------------------+------------------+
|  count|              10527|              10527|              8095|
|   mean|  51.72851271273856|-1.2609543790253606| 190.1229153798641|
| stddev|0.08253548493439644|0.12902241722679658|102.66380337417593|
|    min|          51.550175|        

In [39]:
save_single_csv(loc_clean,CLEANED_DIR, "location_pings.csv", tracker_loc.final_rows)


Saved -> D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned\location_pings.csv (10,527 rows)


In [40]:
disruptions_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("situation_number", StringType(), True),
    StructField("creation_time", StringType(), True),
    StructField("participant_ref", StringType(), True),
    StructField("version", StringType(), True),
    StructField("progress", StringType(), True),
    StructField("misc_reason", StringType(), True),
    StructField("planned", StringType(), True),
    StructField("validity_start", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("description", StringType(), True),
])

try:
    disruptions_raw = spark.read.csv(
        str(PROCESSED_DIR / "disruptions.csv"), header=True, schema=disruptions_schema
    )
except Exception as e:
    raise SystemExit(f"Failed to load disruptions.csv: {e}")

tracker_dis = DQTracker("disruptions")
tracker_dis.original_rows = disruptions_raw.count()
print(f"Loaded {tracker_dis.original_rows:,} rows")


Loaded 598 rows


In [41]:
print(f"Row count: {disruptions_raw.count():,}")
print(f"Columns:   {disruptions_raw.columns}")
disruptions_raw.printSchema()
disruptions_raw.show(5, truncate=False)


Row count: 598
Columns:   ['source_file', 'situation_number', 'creation_time', 'participant_ref', 'version', 'progress', 'misc_reason', 'planned', 'validity_start', 'summary', 'description']
root
 |-- source_file: string (nullable = true)
 |-- situation_number: string (nullable = true)
 |-- creation_time: string (nullable = true)
 |-- participant_ref: string (nullable = true)
 |-- version: string (nullable = true)
 |-- progress: string (nullable = true)
 |-- misc_reason: string (nullable = true)
 |-- planned: string (nullable = true)
 |-- validity_start: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- description: string (nullable = true)

+-----------+------------------------------------+------------------------+---------------+-------+--------+-----------+-------+--------------+--------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------

In [42]:
dis_std = standardize_column_names(disruptions_raw)
dis_std = trim_string_columns(dis_std)

missing_value_report(dis_std, "disruptions (raw)")
duplicate_count(dis_std, ["situation_number", "version"], "disruptions")

print("\nDistinct 'progress' values before standardisation (case check):")
dis_std.select("progress").distinct().show()



Missing value report -- disruptions (raw) (of 598 rows):
  situation_number              :        102 (17.1%)
  creation_time                 :        122 (20.4%)
  participant_ref               :        133 (22.2%)
  version                       :        139 (23.2%)
  progress                      :        139 (23.2%)
  misc_reason                   :        249 (41.6%)
  planned                       :        141 (23.6%)
  validity_start                :        590 (98.7%)
  summary                       :        142 (23.7%)
  description                   :        142 (23.7%)
Duplicate check -- disruptions on ['situation_number', 'version']: 103 duplicate row(s) found (598 -> 495)

Distinct 'progress' values before standardisation (case check):
+--------------------+
|            progress|
+--------------------+
|                 76A|
|                   8|
|                  50|
|                open|
|                  8A|
|                 52A|
|                 621|
|         

In [43]:
dis_clean = dis_std.dropDuplicates(["situation_number", "version"])
tracker_dis.after_dedup_rows = dis_clean.count()

dis_clean = dis_clean.filter(
    F.col("situation_number").isNotNull() & F.col("creation_time").isNotNull()
)
dis_clean = dis_clean.fillna({"misc_reason": "unspecified"})
tracker_dis.after_missing_rows = dis_clean.count()

dis_clean = dis_clean.withColumn(
    "creation_ts", F.try_to_timestamp("creation_time")
).withColumn(
    "validity_start_ts", F.try_to_timestamp("validity_start")
).withColumn(
    "progress", F.lower(F.col("progress"))
).withColumn(
    "planned", (F.lower(F.col("planned")) == "true").cast(BooleanType())
).withColumn(
    "version", F.col("version").cast(IntegerType())
)
tracker_dis.final_rows = dis_clean.count()
print(f"Rows after cleaning: {tracker_dis.final_rows:,}")


Rows after cleaning: 476


In [44]:
missing_value_report(dis_clean, "disruptions (cleaned)")
duplicate_count(dis_clean, ["situation_number", "version"], "disruptions (cleaned)")

print("\nDisruption reason breakdown:")
dis_clean.groupBy("misc_reason").count().orderBy(F.desc("count")).show()

dq_trackers.append(tracker_dis)
print(tracker_dis.as_row())



Missing value report -- disruptions (cleaned) (of 476 rows):
  participant_ref               :         11 (2.3%)
  version                       :         18 (3.8%)
  progress                      :         17 (3.6%)
  planned                       :         19 (4.0%)
  validity_start                :        468 (98.3%)
  summary                       :         20 (4.2%)
  description                   :         20 (4.2%)
  creation_ts                   :         27 (5.7%)
  validity_start_ts             :        476 (100.0%)
Duplicate check -- disruptions (cleaned) on ['situation_number', 'version']: 0 duplicate row(s) found (476 -> 476)

Disruption reason breakdown:
+--------------------+-----+
|         misc_reason|count|
+--------------------+-----+
|           roadworks|  218|
|         unspecified|  127|
|          roadClosed|   52|
|  insufficientDemand|   26|
|        specialEvent|   20|
|      routeDiversion|    8|
|            incident|    7|
|             unknown|    6|
|  

In [45]:
save_single_csv(dis_clean, CLEANED_DIR, "disruptions.csv", tracker_dis.final_rows)


Saved -> D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned\disruptions.csv (476 rows)


In [46]:
fares_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("publication_timestamp", StringType(), True),
    StructField("participant_ref", StringType(), True),
    StructField("operator_ref", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("from_date", StringType(), True),
    StructField("description", StringType(), True),
])

try:
    fares_raw = spark.read.csv(
        str(PROCESSED_DIR / "fares.csv"), header=True, schema=fares_schema
    )
except Exception as e:
    raise SystemExit(f"Failed to load fares.csv: {e}")

tracker_fares = DQTracker("fares")
tracker_fares.original_rows = fares_raw.count()
print(f"Loaded {tracker_fares.original_rows:,} rows")


Loaded 3,435 rows


In [47]:
print(f"Row count: {fares_raw.count():,}")
print(f"Columns:   {fares_raw.columns}")
fares_raw.printSchema()
fares_raw.show(5, truncate=False)


Row count: 3,435
Columns:   ['source_file', 'publication_timestamp', 'participant_ref', 'operator_ref', 'line_ref', 'from_date', 'description']
root
 |-- source_file: string (nullable = true)
 |-- publication_timestamp: string (nullable = true)
 |-- participant_ref: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- from_date: string (nullable = true)
 |-- description: string (nullable = true)

+--------------------------------------------------------------------------------------+----------------------------+---------------+------------+--------------------+--------------------+--------------------------------------+
|source_file                                                                           |publication_timestamp       |participant_ref|operator_ref|line_ref            |from_date           |description                           |
+----------------------------------------------------------------------------------

In [48]:
fares_std = standardize_column_names(fares_raw)
fares_std = trim_string_columns(fares_std)

missing_value_report(fares_std, "fares (raw)")
duplicate_count(fares_std, ["participant_ref", "line_ref", "from_date"], "fares")



Missing value report -- fares (raw) (of 3,435 rows):
  line_ref                      :        422 (12.3%)
Duplicate check -- fares on ['participant_ref', 'line_ref', 'from_date']: 3,307 duplicate row(s) found (3,435 -> 128)


3307

In [49]:
fares_clean = fares_std.dropDuplicates(["participant_ref", "line_ref", "from_date"])
tracker_fares.after_dedup_rows = fares_clean.count()

fares_clean = fares_clean.filter(
    F.col("participant_ref").isNotNull() & F.col("line_ref").isNotNull()
)
fares_clean = fares_clean.fillna({"description": "No description provided"})
tracker_fares.after_missing_rows = fares_clean.count()

fares_clean = fares_clean.withColumn(
    "from_date_ts", F.try_to_timestamp("from_date")
).withColumn(
    "publication_ts", F.try_to_timestamp("publication_timestamp")
)
tracker_fares.final_rows = fares_clean.count()
print(f"Rows after cleaning: {tracker_fares.final_rows:,}")


Rows after cleaning: 126


In [50]:
missing_value_report(fares_clean, "fares (cleaned)")
duplicate_count(fares_clean, ["participant_ref", "line_ref", "from_date"], "fares (cleaned)")

print("\nFare publications per operator:")
fares_clean.groupBy("operator_ref").count().orderBy(F.desc("count")).show()

dq_trackers.append(tracker_fares)
print(tracker_fares.as_row())



Missing value report -- fares (cleaned) (of 126 rows):
  No missing values found.
Duplicate check -- fares (cleaned) on ['participant_ref', 'line_ref', 'from_date']: 0 duplicate row(s) found (126 -> 126)

Fare publications per operator:
+------------+-----+
|operator_ref|count|
+------------+-----+
|    noc:SCGL|   84|
|    noc:SCOX|   42|
+------------+-----+

{'Dataset': 'fares', 'Original Rows': 3435, 'Final Rows': 126, 'Duplicates Removed': 3307, 'Missing Handled': 2, 'Invalid Removed': 0}


In [51]:
save_single_csv(fares_clean, CLEANED_DIR, "fares.csv", tracker_fares.final_rows)


Saved -> D:\Fourth Semester\Big Data\Project\Bus_Delay_Prediction\data\cleaned\fares.csv (126 rows)


In [52]:
summary_rows = [t.as_row() for t in dq_trackers]
summary_df = pd.DataFrame(summary_rows)
summary_df["% Retained"] = (
    100 * summary_df["Final Rows"] / summary_df["Original Rows"]
).round(1)

print(summary_df.to_string(index=False))


                   Dataset  Original Rows  Final Rows  Duplicates Removed  Missing Handled  Invalid Removed  % Retained
           timetable_stops          20570        8895               11675                0                0        43.2
timetable_vehicle_journeys          35997       35997                   0                0                0       100.0
      timetable_stop_times        1126331      926481              199850                0                0        82.3
            location_pings          21048       10527               10521                0                0        50.0
               disruptions            598         476                 103               19                0        79.6
                     fares           3435         126                3307                2                0         3.7


In [53]:
spark.stop()
print("Spark session stopped. Notebook 01 complete.")

Spark session stopped. Notebook 01 complete.


In [54]:
import os
os.getcwd()

'C:\\Users\\Acer\\Bus_Delay_Prediction'